
# Interactive Grid Comparison Analysis
This notebook enables interactive comparison of Synthetic Grid Versions against Real Grid Data.

**Features:**
- **Interactive Plotting**: Choose metrics and plot types via dropdowns.
- **Statistical Analysis**: Kolmogorov-Smirnov (KS) tests to quantify distribution similarity.
- **Geographic Analysis**: Visualize grid structures on map.
- **Data Loading**: Auto-detects Real Grid CSVs or allows path configuration.


In [1]:

import pandas as pd
import psycopg2
import os
import sys
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display
from scipy import stats

# Add project root to path for imports if running locally without install
project_root = Path("../../").resolve()
sys.path.append(str(project_root / "src"))

# Import Refactored Modules
from pylovo.plotting.validation import metric_validation as plotting
from pylovo.plotting.validation import geo_validation
from pylovo.database import database_client

load_dotenv(project_root / ".env")

# Configuration
PLZ = 91301
REAL_METRICS_PATH = Path("results/comparison_metrics.csv")
if not REAL_METRICS_PATH.exists():
    found = list(Path(".").rglob("comparison_metrics.csv"))
    if found:
        REAL_METRICS_PATH = found[0]


In [2]:

def get_db_connection():
    try:
        conn = psycopg2.connect(
            dbname=os.environ.get("DBNAME"),
            user=os.environ.get("DBUSER"),
            password=os.environ.get("PASSWORD"),
            host=os.environ.get("HOST"),
            port=os.environ.get("PORT")
        )
        return conn
    except Exception as e:
        print(f"DB Connection Failed: {e}")
        return None

def load_real_metrics(csv_path):
    if not csv_path.exists():
        print(f"WARNING: Real metrics CSV not found at {csv_path}")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    # ROBUST FILTERING
    real_df = df[df['source'].str.contains('Real', case=False, na=False)]
    if real_df.empty:
        print("Warning: CSV loaded but no 'Real' source found inside.")
    return real_df

def load_synthetic_metrics(conn, plz, versions=None):
    if conn is None: return pd.DataFrame()
    query = f'''
    SELECT 
        gr.version_id, gr.plz, gr.kcid, gr.bcid, gr.grid_result_id,
        cp.no_households, cp.no_household_equ, cp.max_no_of_households_of_a_branch,
        cp.no_branches, cp.max_power_mw, cp.simultaneous_peak_load_mw,
        cp.transformer_mva, cp.avg_trafo_dis, cp.max_trafo_dis,
        cp.max_vsw_of_a_branch, cp.cable_length_km, cp.no_house_connections,
        cp.resistance, cp.reactance,
        cp.no_connection_buses, cp.no_house_connections_per_branch, cp.no_households_per_branch, cp.house_distance_km, cp.cable_len_per_house
    FROM pylovo.grid_result gr
    JOIN pylovo.clustering_parameters cp ON gr.grid_result_id = cp.grid_result_id
    WHERE gr.plz = {plz}
    '''
    if versions:
        if len(versions) == 1:
            query += f" AND gr.version_id = '{versions[0]}'"
        else:
            query += f" AND gr.version_id IN {tuple(versions)}"
            
    try:
        df = pd.read_sql(query, conn)
        if not df.empty:
            df['source'] = 'Synthetic ' + df['version_id']
            # [Auto-Fix] Align Columns
            df['ratio'] = df['resistance'] / df['reactance'].replace(0, np.nan)
            df.rename(columns={'max_vsw_of_a_branch': 'vsw_per_branch'}, inplace=True)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

def get_available_versions(conn):
    if conn is None: return pd.DataFrame()
    try:
        return pd.read_sql("SELECT DISTINCT version_id, version_comment FROM pylovo.version", conn)
    except:
        return pd.DataFrame()



In [3]:

conn = get_db_connection()
versions_df = get_available_versions(conn)
print("Available Versions:")
display(versions_df)

# Load Real
df_real = load_real_metrics(REAL_METRICS_PATH)
if not df_real.empty:
    print(f"Loaded {len(df_real)} Real Grids.")
else:
    print("FAILED TO LOAD REAL GRIDS.")

# Load Synthetic
version_ids = versions_df['version_id'].tolist() if not versions_df.empty else ['1'] 
df_synth = load_synthetic_metrics(conn, PLZ, versions=version_ids)

# Combine
df_all = pd.concat([df_real, df_synth], ignore_index=True)
print(f"Total Grids Loaded: {len(df_all)}")


Available Versions:


,version_id,version_comment
0,2,swf_equipment
1,1,base version


Loaded 84 Real Grids.
Total Grids Loaded: 212



## interactive Metric Distribution
Select a metric to compare distributions via Boxplots.


In [4]:
from IPython.display import display

numeric_cols = df_all.select_dtypes(include=np.number).columns.tolist()
# Filter out IDs
metrics = [c for c in numeric_cols if c not in ['plz', 'kcid', 'bcid', 'grid_result_id', 'version_id']]

# Robust Pattern V2: Method-based clear and display()
out_box = widgets.Output()

def update_boxplot(change):
    metric = change['new'] if isinstance(change, dict) else change
    out_box.clear_output(wait=True)
    with out_box:
        fig = plotting.plot_comparison_distribution_plotly(df_all, metric, title=f"Distribution of {metric}")
        display(fig)

dropdown_box = widgets.Dropdown(options=metrics, value='transformer_mva', description='Metric:')
dropdown_box.observe(update_boxplot, names='value')

# Initialize
update_boxplot(dropdown_box.value)

display(dropdown_box, out_box)


Dropdown(description='Metric:', index=9, options=('no_connection_buses', 'no_branches', 'no_house_connections'…

Output()


## Interactive Histograms
Overlap histograms to analyze distribution shape and tail behavior.


In [ ]:
# Robust Pattern V2
out_hist = widgets.Output()

def update_hist(change):
    metric = change['new'] if isinstance(change, dict) else change
    out_hist.clear_output(wait=True)
    with out_hist:
        fig = plotting.plot_comparison_histogram_plotly(df_all, metric, title=f"Histogram of {metric}")
        display(fig)

dropdown_hist = widgets.Dropdown(options=metrics, value='cable_length_km', description='Metric:')
dropdown_hist.observe(update_hist, names='value')

# Initialize
update_hist(dropdown_hist.value)

display(dropdown_hist, out_hist)


Dropdown(description='Metric:', index=12, options=('no_connection_buses', 'no_branches', 'no_house_connections…

Output()


## Geographic Visualization (GIS)
Visualize the grids on a map.


In [6]:

# Geographic Visualization (GIS)
# ------------------------------
# Option 1: Overview of Transformers (Moderate load)
# print('Plotting Transformers...')
# fig_overview = geo_validation.plot_trafo_on_map(PLZ)
# fig_overview.show()

# Option 2: Full Grid Visualization (Heavy load ~1-3 mins)
# Includes all lines and buses. Recommended to run only if needed.
print('To plot full grid, uncomment the lines below:')
# fig_full = geo_validation.plot_all_grids_detailed(PLZ)
# if fig_full: fig_full.show()


To plot full grid, uncomment the lines below:


In [9]:
# # Inspect specific grid (Example: First entry from loaded data)
# if not df_synth.empty:
#     sample_grid = df_synth.iloc[0]
#     kcid = sample_grid['kcid']
#     bcid = sample_grid['bcid']
#     print(f"Zooming in on Grid KCID={kcid}, BCID={bcid}")
#     fig_zoom = geo_validation.plot_grid_on_map_plotly(PLZ, int(kcid), int(bcid), title=f"Grid {kcid}-{bcid}")
#     if fig_zoom:
#         fig_zoom.show()


## Statistical Comparison (KS-Test)
Mathematically quantify the difference between Real and Synthetic distributions.


In [8]:

def calculate_ks_stats(metric):
    if df_real.empty:
        print("Cannot calculate stats: Missing Real Data")
        return
        
    real_vals = df_real[metric].dropna()
    results = []
    
    for ver in df_synth['version_id'].unique():
        synth_vals = df_synth[df_synth['version_id'] == ver][metric].dropna()
        if synth_vals.empty: continue
        
        stat, pval = stats.ks_2samp(real_vals, synth_vals)
        results.append({
            "Version": ver,
            "Metric": metric,
            "KS Statistic": round(stat, 4),
            "P-Value": round(pval, 4), 
            "Real Mean": round(real_vals.mean(), 2),
            "Synth Mean": round(synth_vals.mean(), 2)
        })
    
    if not results: return
    res_df = pd.DataFrame(results)
    display(res_df.style.background_gradient(subset=['KS Statistic'], cmap='Reds'))

widgets.interact(calculate_ks_stats, metric=widgets.Dropdown(options=metrics, value='transformer_mva', description='Metric:'));


interactive(children=(Dropdown(description='Metric:', index=9, options=('no_connection_buses', 'no_branches', …